# **ワインデータセットを用いた機械学習入門**

## 概要

ポルトガルワインのデータセットを使ってより高度な機械学習手法を実習します。

> 1. **一般化線形モデル（GLM）** — *ポアソン回帰による品質スコアの予測*
> 1. **正規化線型回帰** — *多重共線性への対処*
> 1. **XGBoost（勾配ブースティング）** — *品質の予測と特徴量重要度の可視化*
> 1. **ニューラルネットワーク** — *生成AIへの第一歩*
> 1. **階層的クラスタリング** — *ラベルなしで赤ワインと白ワインを識別*
> 1. **主成分分析** — *因子負荷量を見ながら特徴量について改めて考える*

<img width="600" alt="Vinho Verde wine" src="https://github.com/MotomuMatsui/lecture/blob/main/images/wine.jpg?raw=true">
by ChatGPT 5.5

## データセットについて
ポルトガルの「Vinho Verde」ワインについて、赤ワイン約1,600本・白ワイン約4,900本の以下の情報が記録されています。

| 列名 | 意味 |
| --- | --- |
| `fixed acidity` | 不揮発性酸度 |
| `volatile acidity` | 揮発性酸度(高いと酢っぽくなる) |
| `citric acid` | クエン酸 |
| `residual sugar` | 残糖 |
| `chlorides` | 塩化物 |
| `free sulfur dioxide` | 遊離亜硫酸 |
| `total sulfur dioxide` | 総亜硫酸 |
| `density` | 密度 |
| `pH` | pH |
| `sulphates` | 硫酸塩 |
| `alcohol` | アルコール度数 |
| `quality` | 品質スコア(0〜10の整数、官能評価) |

### Citation
- Aeberhard S and Forina M (1992) Wine. UCI Machine Learning Repository. https://doi.org/10.24432/C5PC7J.
- Aeberhard S, Coomans D, de Vel O (1994) **Comparative analysis of statistical pattern recognition methods in high dimensional settings**,
*Pattern Recognition*, 27(8):1065-1077

## 0. 必要ライブラリのインストールと設定

In [ ]:
# 初回のみ実行してください
%pip install scikit-learn seaborn matplotlib pandas scipy statsmodels matplotlib-fontja xgboost ptitprince

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# ---- 日本語フォントの設定(文字化け対策) ----
# 順序が重要:seaborn のテーマを「先に」設定すると rcParams を初期化するので、
# その「あと」で matplotlib-fontja を import すれば font.family が上書きされない。
sns.set_style("whitegrid")
print("kernel python :", sys.executable)
try:
    import matplotlib_fontja  # noqa: F401
    # 念のため font.sans-serif の先頭にも IPAexGothic を入れておく
    # (将来 sns.set_theme() などで font.family が 'sans-serif' に戻されても CJK が出るように)
    _sans = [f for f in plt.rcParams["font.sans-serif"] if f != "IPAexGothic"]
    plt.rcParams["font.sans-serif"] = ["IPAexGothic"] + _sans
    print("matplotlib-fontja: OK")
except ImportError as e:
    print(f"matplotlib-fontja import 失敗: {e}")
    print("→ 上のセル(%pip install ...)を先に実行してください。")
    from matplotlib import font_manager
    _candidates = [
        "IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
        "TakaoGothic", "VL Gothic", "Hiragino Sans", "Yu Gothic", "Meiryo",
    ]
    _available = {f.name for f in font_manager.fontManager.ttflist}
    _picked = next((n for n in _candidates if n in _available), None)
    if _picked:
        plt.rcParams["font.family"] = _picked
        plt.rcParams["font.sans-serif"] = [_picked] + plt.rcParams["font.sans-serif"]
        print(f"フォールバックで {_picked!r} を使用します。")
    else:
        print("利用可能な CJK フォントが見つかりません。")
plt.rcParams["axes.unicode_minus"] = False
print("font.family    :", plt.rcParams["font.family"])
print("font.sans-serif:", plt.rcParams["font.sans-serif"][:3], "...")

## 1. データの取得と確認

UCI のサーバから直接 CSV を読み込みます。赤ワインと白ワインの2ファイルを取得し、`color` 列を付けて結合します。

### ダウンロード

In [ ]:
URL_RED = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
URL_WHITE = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

red = pd.read_csv(URL_RED, sep=";")
white = pd.read_csv(URL_WHITE, sep=";")

# 色のラベルを付与して結合
red["color"] = "red"
white["color"] = "white"
df = pd.concat([red, white], ignore_index=True)

# 列名にスペースが含まれていると扱いにくいので、アンダースコアに置換
df.columns = df.columns.str.replace(" ", "_")

print(f"赤ワイン: {len(red)} 本")
print(f"白ワイン: {len(white)} 本")
print(f"合計    : {len(df)} 本")
df.head()

### 各列の型と欠損数

In [ ]:
df.info()

### 基本統計量

In [ ]:
df.describe()

## 2. 探索的データ解析（EDA）

### Rain Cloud Plot

色（red/white）で塗り分けることで、特徴量ごとに「赤と白で分布がどれだけ違うか」観察してみましょう。

In [ ]:
import warnings
import ptitprince as pt

# 品質・色以外の数値カラムを「特徴量」として扱う
features = [c for c in df.columns if c not in ["quality", "color"]]

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
# ptitprince 内部の sns.boxplot/stripplot 呼び出しが古い API のため FutureWarning が出る。
# この呼び出し中だけ無視
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", category=FutureWarning, module="ptitprince.*"
    )
    for ax, feat in zip(axes.flat, features):
        pt.RainCloud(
            x="color", y=feat, data=df,
            palette={"red": "#b22222", "white": "#daa520"},
            bw=.2, width_viol=.6, ax=ax, orient="v",
            point_size=1.2, alpha=.4,
        )
        ax.set_title(feat)
        ax.set_xlabel("")

# 余ったサブプロットを非表示に
for ax in axes.flat[len(features):]:
    ax.set_visible(False)

plt.suptitle("Rain Cloud Plot（赤ワイン vs 白ワイン）",
             y=1.00, fontsize=14)
plt.tight_layout()
plt.show()

### 品質スコアの分布と特徴量との関係

In [ ]:
# 品質スコアのヒストグラム（赤白別）
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x="quality", hue="color",
              palette={"red": "#b22222", "white": "#daa520"}, ax=ax)
ax.set_title("品質スコアの分布（赤ワイン vs 白ワイン）")
plt.tight_layout()
plt.show()

In [ ]:
# 品質と各特徴量の関係を箱ひげ図で
features = [c for c in df.columns if c not in ["quality", "color"]]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, feat in zip(axes.flat, features):
    sns.boxplot(data=df, x="quality", y=feat, ax=ax,
                color="lightsteelblue", fliersize=2)
    ax.set_title(feat)
# 余ったサブプロットを非表示に
for ax in axes.flat[len(features):]:
    ax.set_visible(False)
plt.suptitle("品質スコアと各特徴量の関係", y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# 相関行列のヒートマップ
plt.figure(figsize=(10, 8))
corr = df[features + ["quality"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": 0.8})
plt.title("特徴量間の相関行列")
plt.tight_layout()
plt.show()

> **Q1：qualityと強い相関が見られた特徴量について、その理由を考察してみよう**

## 3. 一般化線形モデル（ポアソン回帰）で品質を予測

品質スコアは0〜10の整数値（計数データ）なので、ポアソン回帰を行います。

### データの準備

In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

X = df[features].copy()
y = df["quality"].copy()

# 標準化(係数の大小比較がしやすくなる)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)

### モデルの構築

In [ ]:
# 訓練・テスト分割
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# ポアソン回帰(GLM: family=Poisson, link=log)
X_train_sm = sm.add_constant(X_train)
glm = sm.GLM(y_train, X_train_sm, family=sm.families.Poisson())
result = glm.fit()
print(result.summary())

### モデルの検証

In [ ]:
# テストデータでの予測
X_test_sm = sm.add_constant(X_test)
y_pred = result.predict(X_test_sm)

print(f"MAE  (平均絶対誤差): {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE (二乗平均平方根誤差): {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")

### 係数の可視化

In [ ]:
# 係数の可視化(標準化済みなので大小比較が可能)
coefs = result.params.drop("const").sort_values()

plt.figure(figsize=(8, 5))
colors = ["crimson" if v < 0 else "steelblue" for v in coefs]
coefs.plot(kind="barh", color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("係数(標準化後)")
plt.title("ポアソン回帰の係数（青=品質に正の効果、赤=負の効果）")
plt.tight_layout()
plt.show()

> **Q2：有意ではないのに、大きな絶対値の係数を持つ変数は何を意味しているでしょうか？**

## 4. 正則化線形回帰（Ridge/Lasso）で多重共線性に対処する

上のポアソン回帰で`random_state=42`の数値を何度か変えて実行してみてください。例えば`density`や`alcohol`の係数が大きくブレるのが分かると思います。
これは特徴量同士が強く相関している（**多重共線性**）せいで、係数の推定が不安定になるからです。

**正則化**は、係数の大きさそのものに罰則を加えることでこの不安定さを抑える手法です。

- **Ridge (L2 正則化)**: 係数の二乗和に罰則 → 大きな係数を全体的に縮める
- **Lasso (L1 正則化)**: 係数の絶対値の和に罰則 → 一部の係数を**ちょうどゼロ**にして変数選択を行う

ここではscikit-learnの`RidgeCV`/`LassoCV`を使い、**交差検証で罰則の強さ`alpha`を自動選択**します（なお、scikit-learnのRidge/Lassoはガウス回帰なのでポアソン回帰とは厳密には別物です。係数の方向性の比較を主眼とします）。

### ラッソとリッジで特徴量選択

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV, lasso_path

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

alphas = np.logspace(-3, 2, 50)

ridge = RidgeCV(alphas=alphas, cv=5).fit(X_train, y_train)
lasso = LassoCV(alphas=alphas, cv=5, max_iter=20000, random_state=42).fit(X_train, y_train)

print(f"Ridge: 選ばれた alpha = {ridge.alpha_:.4f}")
print(f"Lasso: 選ばれた alpha = {lasso.alpha_:.4f}")
print(f"Lasso でゼロになった係数の数: {(lasso.coef_ == 0).sum()} / {len(features)}")

y_pred_ridge = ridge.predict(X_test)
y_pred_lasso = lasso.predict(X_test)

print()
print(f"Ridge  MAE: {mean_absolute_error(y_test, y_pred_ridge):.3f}, "
      f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.3f}")
print(f"Lasso  MAE: {mean_absolute_error(y_test, y_pred_lasso):.3f}, "
      f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.3f}")

### モデルの比較

In [ ]:
# 係数の比較:GLM (Poisson, 正則化なし) vs Ridge vs Lasso
coef_df = pd.DataFrame({
    "GLM (Poisson)": result.params.drop("const"),
    "Ridge":          pd.Series(ridge.coef_, index=features),
    "Lasso":          pd.Series(lasso.coef_, index=features),
})
# GLM の係数順に並べる
coef_df = coef_df.loc[coef_df["GLM (Poisson)"].sort_values().index]

ax = coef_df.plot(
    kind="barh", figsize=(9, 6),
    color=["#888888", "steelblue", "crimson"],
    width=0.8,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("係数（標準化後）")
ax.set_title("3つのモデルの係数比較（ポアソン回帰, リッジ回帰, ラッソ回帰）")
plt.tight_layout()
plt.show()

In [ ]:
# 3モデルの精度比較表
comparison_reg = pd.DataFrame({
    "モデル": ["GLM (Poisson)", "Ridge", "Lasso"],
    "MAE":  [
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, y_pred_ridge),
        mean_absolute_error(y_test, y_pred_lasso),
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred)),
        np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
        np.sqrt(mean_squared_error(y_test, y_pred_lasso)),
    ],
})
print(comparison_reg.to_string(index=False))

### リッジ回帰を深掘りしてみる

In [ ]:
# Ridge も同じ alpha グリッドで「係数の正則化パス」と「CV MSE 曲線」を描く。
# RidgeCV は LassoCV と違って alpha 別の CV 結果を保存しないので、ここで手動計算する。
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# (a) 係数の正則化パス:各 alpha で Ridge を当てはめて係数を取り出す
ridge_coefs_path = np.array([
    Ridge(alpha=a).fit(X_train, y_train).coef_
    for a in alphas
]).T   # shape: (n_features, n_alphas)

# (b) alpha ごとの 5-fold CV MSE
ridge_mse_per_fold = np.array([
    -cross_val_score(Ridge(alpha=a), X_train, y_train,
                     scoring="neg_mean_squared_error", cv=5)
    for a in alphas
])
ridge_mse_mean = ridge_mse_per_fold.mean(axis=1)
ridge_mse_std  = ridge_mse_per_fold.std(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# (左) Ridge の係数パス:Lasso と違って 0 にはならず滑らかに縮む
for i, feat in enumerate(features):
    axes[0].plot(alphas, ridge_coefs_path[i], label=feat)
axes[0].set_xscale("log")
axes[0].axvline(ridge.alpha_, color="black", linestyle="--", linewidth=0.8,
                label=f"CV選択 α={ridge.alpha_:.3f}")
axes[0].invert_xaxis()   # 左から右に正則化が強くなる方向に
axes[0].set_xlabel("alpha (大きいほど正則化が強い)")
axes[0].set_ylabel("係数")
axes[0].set_title("Ridge の正則化パス:係数は 0 にならず滑らかに縮む")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

# (右) Ridge の CV MSE 曲線
axes[1].plot(alphas, ridge_mse_mean, color="crimson", linewidth=1.5,
             label="平均 MSE (5-fold CV)")
axes[1].fill_between(alphas, ridge_mse_mean - ridge_mse_std,
                     ridge_mse_mean + ridge_mse_std,
                     color="crimson", alpha=0.2, label="±1σ")
axes[1].set_xscale("log")
axes[1].axvline(ridge.alpha_, color="black", linestyle="--", linewidth=0.8,
                label=f"CV選択 α={ridge.alpha_:.3f}")
axes[1].invert_xaxis()
axes[1].set_xlabel("alpha (大きいほど正則化が強い)")
axes[1].set_ylabel("交差検証 MSE")
axes[1].set_title("Ridge の MSE 曲線:CV が α を選んだ根拠")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

### ラッソ回帰を深掘りしてみる

In [ ]:
# Lasso の正則化パス:alpha を変えると、各係数の値と交差検証 MSE がどう動くか
alphas_path, coefs_path, _ = lasso_path(
    X_train.values, y_train.values, alphas=alphas
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# (左) 係数の正則化パス
for i, feat in enumerate(features):
    axes[0].plot(alphas_path, coefs_path[i], label=feat)
axes[0].set_xscale("log")
axes[0].axvline(lasso.alpha_, color="black", linestyle="--", linewidth=0.8,
                label=f"CV選択 α={lasso.alpha_:.3f}")
axes[0].invert_xaxis()   # 左から右に正則化が強くなる方向に
axes[0].set_xlabel("alpha (大きいほど正則化が強い)")
axes[0].set_ylabel("係数")
axes[0].set_title("Lassoの正則化パス:強いαでは多くの係数が0に縮む")
axes[0].legend(loc="lower left", fontsize=7.5)

# (右) alpha vs 交差検証 MSE(LassoCV が α を選んだ根拠)
# lasso.mse_path_ shape = (n_alphas, n_folds), lasso.alphas_ は降順で同じ alphas
mse_mean = lasso.mse_path_.mean(axis=1)
mse_std  = lasso.mse_path_.std(axis=1)

axes[1].plot(lasso.alphas_, mse_mean, color="steelblue", linewidth=1.5,
             label="平均 MSE (5-fold CV)")
axes[1].fill_between(lasso.alphas_, mse_mean - mse_std, mse_mean + mse_std,
                     color="steelblue", alpha=0.2, label="±1σ")
axes[1].set_xscale("log")
axes[1].axvline(lasso.alpha_, color="black", linestyle="--", linewidth=0.8,
                label=f"CV選択 α={lasso.alpha_:.3f}")
axes[1].invert_xaxis()
axes[1].set_xlabel("alpha (大きいほど正則化が強い)")
axes[1].set_ylabel("交差検証 MSE")
axes[1].set_title("LassoのMSE曲線:CVがαを選んだ根拠")
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.show()

> **Q3：正則化の本質的なメリットとは何でしょうか？**

## 5. XGBoost（勾配ブースティング）で品質を予測

ランダムフォレストと並んで強力な学習器が**勾配ブースティング決定木**です。
弱い決定木を順番に積み重ね、前の木が誤った部分を次の木が補正するように学習します。
ここでは代表的な実装である**XGBoost**を適用してみましょう。

### モデルの構築

In [ ]:
from xgboost import XGBRegressor

X = df[features]
y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print(f"XGBoost")
print(f"  MAE : {mean_absolute_error(y_test, y_pred_xgb):.3f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.3f}")

# GLM との比較表
comparison = pd.DataFrame({
    "モデル": ["GLM (Poisson)", "XGBoost"],
    "MAE":  [mean_absolute_error(y_test, y_pred), mean_absolute_error(y_test, y_pred_xgb)],
    "RMSE": [np.sqrt(mean_squared_error(y_test, y_pred)), np.sqrt(mean_squared_error(y_test, y_pred_xgb))],
})
print("\nモデル比較:")
print(comparison.to_string(index=False))

### 重要度の可視化

In [ ]:
# 特徴量重要度の可視化(XGBoost の feature_importances_ はゲイン正規化)
importance = pd.Series(xgb_model.feature_importances_, index=features).sort_values()

plt.figure(figsize=(8, 5))
importance.plot(kind="barh", color="darkgreen")
plt.xlabel("Feature Importance (gain)")
plt.title("XGBoost による特徴量重要度")
plt.tight_layout()
plt.show()

### 予測値 vs 実測値

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, title in [(axes[0], y_pred, "GLM (Poisson)"),
                         (axes[1], y_pred_xgb, "XGBoost")]:
    ax.scatter(y_test, pred, alpha=0.3, s=15)
    ax.plot([3, 9], [3, 9], "r--", linewidth=1)
    ax.set_xlabel("実測値 (quality)")
    ax.set_ylabel("予測値")
    ax.set_title(title)
    ax.set_xlim(2.5, 9.5)
    ax.set_ylim(2.5, 9.5)

plt.tight_layout()
plt.show()

> **Q4：GLMと比べると、なぜXGBoostはより良い精度が出たのでしょうか？また、それでも極端なquality値に対しては予測がうまくいっていません。それはなぜでしょうか？**

## 6. ニューラルネットワーク（MLP）で品質を予測

最後に、もう一つの非線形モデルとして **多層パーセプトロン（MLP; Multi-Layer Perceptron）** を試します。XGBoostとは別の角度で非線形な関係を捉えられるモデルで、深層学習の最も基本的な形でもあります。

### モデルの構築

In [ ]:
from sklearn.neural_network import MLPRegressor

# 標準化済み特徴量で再分割(GLM/Ridge と同じ random_state=42 の分割)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16),   # 隠れ層3層: 64ユニット → 32ユニット → 16ユニット
    activation="relu",
    solver="adam",
    learning_rate_init=1e-3,
    max_iter=500,
    early_stopping=True,           # 検証損失が改善しなくなったら停止
    validation_fraction=0.1,
    random_state=42,
)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)

print(f"学習が止まった epoch 数: {mlp.n_iter_}")
print(f"MLP  MAE : {mean_absolute_error(y_test, y_pred_mlp):.3f}")
print(f"MLP  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_mlp)):.3f}")

In [ ]:
# 学習曲線:訓練損失と検証スコアの推移
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(mlp.loss_curve_, color="steelblue")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("訓練損失 (MSE)")
axes[0].set_title("訓練損失の推移")
axes[0].grid(alpha=0.3)

if hasattr(mlp, "validation_scores_") and mlp.validation_scores_:
    axes[1].plot(mlp.validation_scores_, color="crimson")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("検証 R² スコア")
    axes[1].set_title("検証スコアの推移（early stopping 用）")
    axes[1].grid(alpha=0.3)
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.show()

### 精度比較

In [ ]:
# 5モデルの精度比較表
comparison_all = pd.DataFrame({
    "モデル": ["GLM (Poisson)", "Ridge", "Lasso", "XGBoost", "MLP (NN)"],
    "MAE":  [
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, y_pred_ridge),
        mean_absolute_error(y_test, y_pred_lasso),
        mean_absolute_error(y_test, y_pred_xgb),
        mean_absolute_error(y_test, y_pred_mlp),
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred)),
        np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
        np.sqrt(mean_squared_error(y_test, y_pred_lasso)),
        np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
        np.sqrt(mean_squared_error(y_test, y_pred_mlp)),
    ],
})
print(comparison_all.to_string(index=False))

# 棒グラフでの可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, color in [(axes[0], "MAE", "steelblue"), (axes[1], "RMSE", "darkorange")]:
    ax.barh(comparison_all["モデル"], comparison_all[metric], color=color)
    ax.set_xlabel(metric)
    ax.set_title(f"5モデルの {metric} 比較(短いほど良い)")
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 予測値 vs 実測値:MLP を含めた4モデルを並べる
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
panels = [
    (axes[0, 0], y_pred,       "GLM (Poisson)"),
    (axes[0, 1], y_pred_ridge, "Ridge"),
    (axes[1, 0], y_pred_xgb,   "XGBoost"),
    (axes[1, 1], y_pred_mlp,   "MLP (NN)"),
]
for ax, pred, title in panels:
    ax.scatter(y_test, pred, alpha=0.3, s=15)
    ax.plot([3, 9], [3, 9], "r--", linewidth=1)
    ax.set_xlabel("実測値 (quality)")
    ax.set_ylabel("予測値")
    ax.set_title(title)
    ax.set_xlim(2.5, 9.5)
    ax.set_ylim(2.5, 9.5)
plt.tight_layout()
plt.show()

> **Q5：「大根を正宗で切る」という言葉通り、このデータセットについてはあまりニューラルネットワークは有効でなかったようです。では、どのようなデータセットに対してなら真価を発揮するのでしょうか？**

## 7. 階層的クラスタリング（教師なし学習）

階層的クラスタリングとヒートマップの組み合わせは、データ解析において、しばしば非常に強力な視覚化手法になります。

### 階層的クラスタリング+ヒートマップの描画

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# 計算量を抑えるため、ランダムに800サンプル抽出してクラスタリング
sample = df.sample(n=800, random_state=42).reset_index(drop=True)

X_sample = sample[features]
X_scaled_sample = StandardScaler().fit_transform(X_sample)
X_scaled_df = pd.DataFrame(X_scaled_sample, columns=features, index=sample.index)

# Ward 法でリンケージを計算(後続セルでも fcluster に使う)
Z = linkage(X_scaled_sample, method="ward")

# 行注釈の細い色帯:本当の色ラベル(クラスタリングには使っていない、答え合わせ用)
row_colors = sample["color"].map({"red": "#b22222", "white": "#daa520"})
row_colors.name = "Red wine or White wine"

# clustermap = 行(サンプル)のデンドログラム + 列(特徴量)のデンドログラム
#              + 標準化済み特徴量のヒートマップ
g = sns.clustermap(
    X_scaled_df,
    row_linkage=Z,          # 上で計算した Ward 法のリンケージを使う
    col_cluster=True,       # 特徴量側もクラスタリング
    cmap="vlag", center=0, vmin=-3, vmax=3,
    row_colors=row_colors,
    figsize=(11, 10),
    xticklabels=True, yticklabels=False,
    dendrogram_ratio=(0.18, 0.12),
    colors_ratio=0.02,
    cbar_pos=(0.02, 0.82, 0.025, 0.15),
    cbar_kws={"label": "z-score"},
)
g.ax_heatmap.set_xlabel("特徴量（標準化済み）")
g.ax_heatmap.set_ylabel(f"サンプル(n={len(sample)})")
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")
g.figure.suptitle(
    "階層的クラスタリング（Ward法）+ 各特徴量のヒートマップ\n"
    "（左の色帯=本当の色ラベル。デンドログラムの2大分枝とほぼ対応）",
    y=1.02, fontsize=12,
)
plt.show()

### クラスタリングの精度検証

In [ ]:
# 2クラスタに切り分け
clusters = fcluster(Z, t=2, criterion="maxclust")
sample["cluster"] = clusters

# 答え合わせ:実際の色ラベルとの対応
print("クラスタ vs 色(答え合わせ):")
print(pd.crosstab(sample["cluster"], sample["color"]))

# 一致率を計算
ct = pd.crosstab(sample["cluster"], sample["color"])
best_match = ct.values.max(axis=1).sum() / ct.values.sum()
print(f"\nクラスタを色に最適にマッチさせた場合の一致率: {best_match:.1%}")

In [ ]:
# 可視化:総亜硫酸 × 揮発性酸度 の平面で「クラスタ」と「本当の色」を比較
# (赤白を最もよく分ける2変数の組み合わせ)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(
    data=sample, x="total_sulfur_dioxide", y="volatile_acidity",
    hue="cluster", palette="Set1", ax=axes[0], alpha=0.6, s=25
)
axes[0].set_title("クラスタリング結果（ラベル未使用）")

sns.scatterplot(
    data=sample, x="total_sulfur_dioxide", y="volatile_acidity",
    hue="color", palette={"red": "#b22222", "white": "#daa520"},
    ax=axes[1], alpha=0.6, s=25
)
axes[1].set_title("本当の色ラベル（答え）")

plt.tight_layout()
plt.show()

> **Q6：なぜ赤ワインと白ワインはうまく識別できたのでしょうか？醸造プロセスの違いと関連づけて考察してみましょう**

## 8. 主成分分析（PCA）による次元削減

11個の化学成分（特徴量）を、情報量を保ったまま少数の主成分にまとめます。
上位3個の主成分を**3次元散布図**で表示すれば全体像が一望できるほか、**因子負荷量(loading)**を見ることで「各主成分は元のどの変数を強く反映しているか」を解釈できます。

> **Tip:** Jupyter で 3D 散布図をマウスで回転させたい場合は、セルの先頭に `%matplotlib widget`(要 `jupyter-matplotlib` パッケージ)を書くとインタラクティブモードになります。本ノートブックでは静止画(`%matplotlib inline`、デフォルト)のまま、`view_init` で固定視点を指定しています。

### PCAの実行と3D可視化

In [ ]:
from sklearn.decomposition import PCA

# PCA は標準化した特徴量に対して行うのが基本
X_all = df[features]
X_all_scaled = StandardScaler().fit_transform(X_all)

pca = PCA(n_components=len(features))
scores = pca.fit_transform(X_all_scaled)   # サンプル × 主成分

# 寄与率と累積寄与率
explained = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained)

fig, ax1 = plt.subplots(figsize=(8, 4))
xs = np.arange(1, len(explained) + 1)
ax1.bar(xs, explained, color="steelblue", alpha=0.7, label="寄与率")
ax1.set_xlabel("主成分")
ax1.set_ylabel("寄与率")
ax1.set_xticks(xs)

ax2 = ax1.twinx()
ax2.plot(xs, cum_explained, "o-", color="crimson", label="累積寄与率")
ax2.set_ylabel("累積寄与率")
ax2.set_ylim(0, 1.05)
ax2.axhline(0.8, color="gray", linestyle="--", linewidth=0.8)

ax1.set_title("PCA: 各主成分の寄与率と累積寄与率")
fig.tight_layout()
plt.show()

print("各主成分の寄与率:")
for i, (e, c) in enumerate(zip(explained, cum_explained), 1):
    print(f"  PC{i:2d}: {e:.3f}  (累積 {c:.3f})")

In [ ]:
# PC1 × PC2 × PC3 の3次元空間でサンプルを可視化(色ラベルと品質スコアで色分け)
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  -- '3d' projection を登録

pc_df = pd.DataFrame(scores[:, :3], columns=["PC1", "PC2", "PC3"])
pc_df["color"] = df["color"].values
pc_df["quality"] = df["quality"].values

fig = plt.figure(figsize=(14, 6))

# (左) 赤白の色ラベルで塗り分け
ax0 = fig.add_subplot(1, 2, 1, projection="3d")
for c_label, c_hex in [("red", "#b22222"), ("white", "#daa520")]:
    mask = pc_df["color"] == c_label
    ax0.scatter(
        pc_df.loc[mask, "PC1"], pc_df.loc[mask, "PC2"], pc_df.loc[mask, "PC3"],
        c=c_hex, label=c_label, alpha=0.4, s=10,
    )
ax0.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
ax0.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
ax0.set_zlabel(f"PC3 ({explained[2]*100:.1f}%)")
ax0.set_title("PC1 × PC2 × PC3(赤ワイン vs 白ワイン)")
ax0.legend(loc="upper left")
ax0.view_init(elev=20, azim=45)

# (右) 品質スコアで連続的に色分け
ax1 = fig.add_subplot(1, 2, 2, projection="3d")
sc = ax1.scatter(
    pc_df["PC1"], pc_df["PC2"], pc_df["PC3"],
    c=pc_df["quality"], cmap="viridis", alpha=0.5, s=10,
)
ax1.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
ax1.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
ax1.set_zlabel(f"PC3 ({explained[2]*100:.1f}%)")
ax1.set_title("PC1 × PC2 × PC3(品質スコアで色分け)")
ax1.view_init(elev=20, azim=45)
fig.colorbar(sc, ax=ax1, shrink=0.6, label="quality")

plt.tight_layout()
plt.show()

### 因子負荷量を見てみよう

In [ ]:
# 因子負荷量(loadings)= 各主成分が元の変数とどれくらい相関しているか
# loadings_ij = components_ij * sqrt(eigenvalue_i)
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loadings_df = pd.DataFrame(
    loadings[:, :4],
    index=features,
    columns=[f"PC{i+1}" for i in range(4)],
)

fig = plt.figure(figsize=(15, 6))

# (1) PC1〜PC4 の因子負荷量をヒートマップで
ax0 = fig.add_subplot(1, 2, 1)
sns.heatmap(
    loadings_df, annot=True, fmt=".2f", cmap="coolwarm", center=0,
    cbar_kws={"shrink": 0.8}, ax=ax0,
)
ax0.set_title("各主成分の因子負荷量（PC1〜PC4）")
ax0.set_xlabel("主成分")
ax0.set_ylabel("元の特徴量")

# (2) PC1 × PC2 × PC3 の3Dバイプロット(矢印=変数の負荷量ベクトル)
ax1 = fig.add_subplot(1, 2, 2, projection="3d")
sample_idx = np.random.RandomState(42).choice(len(scores), size=800, replace=False)
ax1.scatter(
    scores[sample_idx, 0], scores[sample_idx, 1], scores[sample_idx, 2],
    c=["#b22222" if c == "red" else "#daa520" for c in df["color"].iloc[sample_idx]],
    alpha=0.15, s=8,
)

# 矢印スケール:散布図の範囲に合わせる
arrow_scale = np.abs(scores[:, :3]).max() * 0.8
for i, feat in enumerate(features):
    x = loadings[i, 0] * arrow_scale
    y = loadings[i, 1] * arrow_scale
    z = loadings[i, 2] * arrow_scale
    ax1.quiver(
        0, 0, 0, x, y, z,
        color="black", alpha=0.7, arrow_length_ratio=0.1, linewidth=1.2,
    )
    ax1.text(x * 1.15, y * 1.15, z * 1.15, feat,
             color="darkblue", fontsize=8, ha="center", va="center")

ax1.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
ax1.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
ax1.set_zlabel(f"PC3 ({explained[2]*100:.1f}%)")
ax1.set_title("3Dバイプロット:サンプル（点）と因子負荷量（矢印）")
ax1.view_init(elev=20, azim=45)

plt.tight_layout()
plt.show()

print("PC1で絶対値が大きい変数（≒ PC1 を主に作っている変数）:")
print(loadings_df["PC1"].abs().sort_values(ascending=False).head(5))
print("\nPC2で絶対値が大きい変数:")
print(loadings_df["PC2"].abs().sort_values(ascending=False).head(5))
print("\nPC3で絶対値が大きい変数:")
print(loadings_df["PC3"].abs().sort_values(ascending=False).head(5))

> **Q7：これまでの解析を一つ一つ振り返りながら、PCAの結果を眺めてみましょう。主成分の構成や因子負荷量と、各結果は整合的でしたか？**

## 9. じっくり考えよう

1. **回帰タスクと分類タスクの違い**
   - 今回、ポアソン回帰を適用しましたが、これはそもそも適切な選択だったのでしょうか。通常の線形回帰（`family=sm.families.Gaussian()`でできます）と比較しながら考えてみよう。

2. **モデルの解釈性と精度の関係**
   - 解釈性という観点から各手法を比較してみましょう。解釈性と精度を同時に向上させることは可能だろうか？

3. **多重共線性**
   - GLMは多重共線性の影響を受けやすい一方で、XGBoostは多重共線性に強いと言われています。それはなぜでしょう。

4. **品質スコアの正体**
   - 品質スコアはそもそもどういう特徴量なのでしょうか。「人間とAI」という大きな観点から考えてみよう。